In [255]:
import json

f = open('../../rag_utility/eval_results/short_answers_0shot_1calls_0_0_bm25_dl_nq_test_concise_eval.json')
zero_evals = json.load(f)
f.close()

_k = 3
_ret = 'bm25'
f = open(f'../../rag_utility/eval_results/short_answers_{_k}shot_1calls_1_0_{_ret}_dl_nq_test_concise_eval.json')
k_evals = json.load(f)
f.close()

f = open(f'../coherence_eval/log_prob_temp_res/nq_test_{_ret}_12.json')
k_probs = json.load(f)
f.close()

# f = open(f'../coherence_eval/log_prob_temp_res/full_context/nq_test_{_ret}_{_k}.json')
# k_probs = json.load(f)
# f.close()


In [256]:
import pandas as pd
import numpy as np

qids_column = list(zero_evals.keys())
zero_f1_column = [zero_evals[qid]['0']['0']['F1'] for qid in qids_column]
k_f1_column = [k_evals[qid]['0']['0']['F1'] for qid in qids_column]
zero_em_column = [zero_evals[qid]['0']['0']['EM'] for qid in qids_column]
k_em_column = [k_evals[qid]['0']['0']['EM'] for qid in qids_column]


eval_df = pd.DataFrame(np.array([zero_f1_column, k_f1_column, zero_em_column, k_em_column]).T, columns=['F1(0)', 'F1(k)', 'EM(0)', 'EM(k)'])
eval_df['qid'] = qids_column
eval_df['utility'] = eval_df['F1(k)'] - eval_df['F1(0)']

try:
    k_prob_column_dict = {}
    for _qid, _probs_temp in k_probs.items():
        k_prob_column_dict.update({_qid: np.mean(list(_probs_temp.values())[:_k])})
    eval_df = eval_df[eval_df.qid.isin(k_prob_column_dict.keys())]
    k_prob_column = [k_prob_column_dict[qid] for qid in eval_df.qid.values]
    eval_df['llm_prob'] = k_prob_column
except:
    print('probs are not calculated')
    
# try:
#     k_prob_column = [k_probs[qid]['full'] for qid in qids_column]
#     eval_df['llm_prob'] = k_prob_column
# except:
#     print('probs are not calculated')

# try:
#     k_prob_column_dict = {}
#     for _qid, _probs_temp in k_probs.items():
#         k_prob_column_dict.update({_qid: _probs_temp})
#     eval_df = eval_df[eval_df.qid.isin(k_prob_column_dict.keys())]
#     k_prob_column = [k_prob_column_dict[qid] for qid in eval_df.qid.values]
#     eval_df['llm_prob'] = k_prob_column
# except:
#     print('probs are not calculated')

print(eval_df.shape)
# eval_df.head(20)

(3610, 7)


In [257]:
from scipy import stats

print('with f1')
print(stats.pearsonr(eval_df['F1(k)'].values, eval_df.llm_prob.values))
print(stats.kendalltau(eval_df['F1(k)'].values, eval_df.llm_prob.values))
print('\nwith utility')
print(stats.pearsonr(eval_df['utility'].values, eval_df.llm_prob.values))
print(stats.kendalltau(eval_df['utility'].values, eval_df.llm_prob.values))

with f1
PearsonRResult(statistic=0.09553294976935074, pvalue=8.866357614108038e-09)
SignificanceResult(statistic=0.0753987572048397, pvalue=2.9899853049390095e-09)

with utility
PearsonRResult(statistic=0.024559762590246517, pvalue=0.14012049427417417)
SignificanceResult(statistic=0.03168244145726421, pvalue=0.009833202481887328)


In [208]:
from tools import coherence_cal

In [98]:
from tools import matrix_tools

In [99]:
import pickle as pkl

res, _, doc_length_dict = coherence_cal.get_res_and_dicts('nq_test', _ret)


KeyboardInterrupt: 

In [ ]:
import math

f = open(f'../coherence_res/bi-directional/nq_test_12_{_ret}.pkl', 'rb')
matrix_book = pkl.load(f)
f.close()

w = 10
coh_dict = coherence_cal.cal_coherence(matrix_book, _k, doc_length_dict, w, math.ceil(w/2), bidirectional=0, distribution='top-heavy')
coh_df = pd.DataFrame(np.array([list(coh_dict.keys()), list(coh_dict.values())]).T, columns=['qid', 'coherence'])

In [ ]:
final_df = eval_df.merge(coh_df, on='qid')
final_df.utility = final_df.utility.astype('float')
final_df.coherence = final_df.coherence.astype('float')
print(final_df.shape)
print(final_df['F1(k)'].mean())

from scipy import stats

print('Correlation with utility')
print('r', stats.pearsonr(final_df.coherence.values, final_df.utility.values))
print('rho', stats.spearmanr(final_df.coherence.values, final_df.utility.values))
print('tau', stats.kendalltau(final_df.coherence.values, final_df.utility.values))

print('\nCorrelation with F1')
print('r', stats.pearsonr(final_df.coherence.values, final_df['F1(k)'].values))
print('rho', stats.spearmanr(final_df.coherence.values, final_df['F1(k)'].values))
print('tau', stats.kendalltau(final_df.coherence.values, final_df['F1(k)'].values))

In [ ]:
final_df

In [ ]:
import qpp_methods

In [ ]:
qpp = qpp_methods.QPP('nq_test')

In [ ]:
import pyterrier_dr

tct_model = pyterrier_dr.TctColBert()
# tct_index = pyterrier_dr.FlexIndex('/mnt/indices/msmarco-passage.tct-hnp.flex')
tct_index = pyterrier_dr.FlexIndex('/mnt/indices/nq_tct_colbert_index_1.flex')

In [ ]:
# spatial_qpp_df = qpp.qpp_in_batch(res[res.qid.isin(res.qid.unique())], 'spatial', _k, tct_model, tct_index)

In [ ]:
# spatial_qpp_df.to_csv(f'./qpp_cache_dense_{_ret}_{_k}.csv', index=False)

In [ ]:
qpp_df = qpp.qpp_in_batch(res, 'nqc', _k, tct_model, tct_index)

In [ ]:
final_df_1 = final_df.merge(qpp_df, on='qid')
final_df_1['coh_spatial'] = final_df_1['coherence'].apply(lambda x: math.log(1+x)) + final_df_1['qpp_estimate'].apply(lambda x: math.log(1+x))
final_df_1.shape

In [ ]:
from scipy import stats

print('Correlation with utility')
print('r', stats.pearsonr(final_df_1.qpp_estimate.values, final_df_1.utility.values))
print('rho', stats.spearmanr(final_df_1.qpp_estimate.values, final_df_1.utility.values))
print('tau', stats.kendalltau(final_df_1.qpp_estimate.values, final_df_1.utility.values))

print('\nCorrelation with F1')
print('r', stats.pearsonr(final_df_1.qpp_estimate.values, final_df_1['F1(k)'].values))
print('rho', stats.spearmanr(final_df_1.qpp_estimate.values, final_df_1['F1(k)'].values))
print('tau', stats.kendalltau(final_df_1.qpp_estimate.values, final_df_1['F1(k)'].values))

In [ ]:
from scipy import stats

print('Correlation with utility')
print('r', stats.pearsonr(final_df_1.coh_spatial.values, final_df_1.utility.values))
print('rho', stats.spearmanr(final_df_1.coh_spatial.values, final_df_1.utility.values))
print('tau', stats.kendalltau(final_df_1.coh_spatial.values, final_df_1.utility.values))

print('\nCorrelation with F1')
print('r', stats.pearsonr(final_df_1.coh_spatial.values, final_df_1['F1(k)'].values))
print('rho', stats.spearmanr(final_df_1.coh_spatial.values, final_df_1['F1(k)'].values))
print('tau', stats.kendalltau(final_df_1.coh_spatial.values, final_df_1['F1(k)'].values))

In [ ]:
import pyterrier as pt

sparse_index = pt.Artifact.from_hf('pyterrier/ragwiki-terrier')

In [ ]:
nq_index_ref = pt.IndexFactory.of('/mnt/indices/BEIR/nq/nq_sparseIndex')

nq_index_ref.getCollectionStatistics().getNumberOfDocuments()

In [ ]:
index_path ='/mnt/indices/msmarco-passage.terrier/'
index_ref = pt.IndexRef.of(index_path)
index = pt.IndexFactory.of(index_ref)
DOC_NUM = index.getCollectionStatistics().getNumberOfDocuments()

In [ ]:
type(sparse_index.path)

In [341]:
import torch

torch.cuda.empty_cache()